# 06 — Association Rules Conditioner: Soft Diagnostic Prior

**Idea:** Mine label co-occurrence patterns from the training set and inject them as a soft
diagnostic prior into the prompt at inference time.

**Mechanism:**
1. Build a pairwise conditional probability table: P(B | A) from training labels.
2. For each test study, scan the indication text for keyword triggers to infer likely labels.
3. Look up high-confidence rules for those labels and format them as a clinical hint:
   > *"Clinical association context: Pleural Effusion co-occurs with Edema in 61% of similar studies (lift=2.1)."*
4. Inject the hint into the prompt before the Findings tag.

Saves `reports/eval_metrics_assoc_rules_{VARIANT}.json` for notebook 04 STEP 7.

**Execution order across notebooks:**
```
04 (core eval)  →  05 (RAG)  →  [this notebook]  →  04 STEP 7 (grand comparison)
```

## STEP 1 — Environment setup

In [ ]:
import os, sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import yaml

matplotlib.rcParams['figure.dpi'] = 120
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR  = REPO_ROOT / 'data' / 'processed'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR    = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(REPO_ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}  {props.total_memory/1e9:.1f} GB')

test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
test_df = test_df[test_df['findings'].notna() & (test_df['findings'].str.strip() != '')].reset_index(drop=True)
references = test_df['findings'].str.strip().tolist()
print(f'Test set : {len(test_df)} studies')

train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
train_df = train_df[train_df['findings'].notna() & (train_df['findings'].str.strip() != '')].reset_index(drop=True)
print(f'Train set: {len(train_df)} studies')

env_file = REPO_ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
    import huggingface_hub
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        huggingface_hub.login(token=hf_token, add_to_git_credential=False)
        print('HF: logged in')

## STEP 2 — Build association rules from training labels

Computes pairwise P(B | A) and lift for all label pairs with prevalence ≥ 1%.  
CPU-only — runs immediately without a GPU.

In [ ]:
from src.data.labels import CHEXBERT_LABELS

train_label_matrix = train_df[CHEXBERT_LABELS].values.astype(float)
n = len(train_label_matrix)

co_matrix   = (train_label_matrix.T @ train_label_matrix) / n
prevalences = train_label_matrix.mean(axis=0)

rules = []
for i, label_a in enumerate(CHEXBERT_LABELS):
    if prevalences[i] < 0.01: continue
    for j, label_b in enumerate(CHEXBERT_LABELS):
        if i == j or prevalences[j] < 0.01: continue
        p_a  = prevalences[i]
        p_ab = co_matrix[i, j]
        conf = p_ab / p_a
        lift = conf / prevalences[j] if prevalences[j] > 0 else 0
        rules.append({
            'antecedent':   label_a,
            'consequent':   label_b,
            'support':      round(float(p_ab),  4),
            'confidence':   round(float(conf),  4),
            'lift':         round(float(lift),  4),
            'p_antecedent': round(float(p_a),   4),
        })

rules_df = pd.DataFrame(rules).sort_values('confidence', ascending=False)
print(f'Total rules      : {len(rules_df)}')
print(f'conf ≥ 0.30      : {(rules_df.confidence >= 0.30).sum()}')
print(f'conf ≥ 0.30 + lift ≥ 1.5: {((rules_df.confidence >= 0.30) & (rules_df.lift >= 1.5)).sum()}')
print()
display(rules_df[rules_df['confidence'] >= 0.30].head(25))

# ── Heat-map of conditional probabilities ─────────────────────────────────────
labels_active = [l for i, l in enumerate(CHEXBERT_LABELS) if prevalences[i] >= 0.01]
active_idx    = [CHEXBERT_LABELS.index(l) for l in labels_active]
conf_matrix   = np.zeros((len(labels_active), len(labels_active)))
for i, ia in enumerate(active_idx):
    for j, ja in enumerate(active_idx):
        if ia != ja and prevalences[ia] > 0:
            conf_matrix[i, j] = co_matrix[ia, ja] / prevalences[ia]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(conf_matrix, cmap='Blues', vmin=0, vmax=0.7, aspect='auto')
ax.set_xticks(range(len(labels_active)))
ax.set_xticklabels(labels_active, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(labels_active)))
ax.set_yticklabels(labels_active, fontsize=8)
for i in range(len(labels_active)):
    for j in range(len(labels_active)):
        if conf_matrix[i, j] > 0:
            ax.text(j, i, f'{conf_matrix[i,j]:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if conf_matrix[i, j] > 0.4 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8, label='P(consequent | antecedent)')
ax.set_xlabel('Consequent'); ax.set_ylabel('Antecedent')
ax.set_title('Label co-occurrence — P(B | A) from training set')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'assoc_rules_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved assoc_rules_heatmap.png'); plt.show()
# ── TF-IDF label prior: retrieve similar studies and compute empirical P(label | indication) ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as _cosine_sim

_TFIDF_K      = 20
_MIN_SIM_TFIDF = 0.05

train_indications_for_rules = train_df['indication'].fillna('').astype(str).tolist()
_rules_tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
_rules_tfidf_matrix = _rules_tfidf.fit_transform(train_indications_for_rules)
print(f'TF-IDF label-prior index: {_rules_tfidf_matrix.shape[0]} docs × {_rules_tfidf_matrix.shape[1]} features')


def retrieve_label_prior(indication: str, k: int = _TFIDF_K) -> dict[str, float] | None:
    """Retrieve top-k similar training studies and return empirical label prevalence.

    Returns None when indication is empty or max similarity is below threshold.
    """
    if not indication.strip():
        return None
    q = _rules_tfidf.transform([indication])
    sims = _cosine_sim(q, _rules_tfidf_matrix).flatten()
    if sims.max() < _MIN_SIM_TFIDF:
        return None
    top_idx = sims.argsort()[::-1][:k]
    subset  = train_label_matrix[top_idx]          # shape (k, 14)
    priors  = dict(zip(CHEXBERT_LABELS, subset.mean(axis=0)))
    return priors


def build_label_prior_hint(priors: dict[str, float], min_prevalence: float = 0.30) -> str:
    """Format empirical label prevalence from retrieved studies as a clinical context hint."""
    relevant = [(label, p) for label, p in priors.items()
                if p >= min_prevalence and label != 'No Finding']
    if not relevant:
        return ''
    relevant.sort(key=lambda x: -x[1])
    lines = ['Clinical context (based on similar studies in training set):']
    for label, p in relevant:
        lines.append(f'  - {label}: present in {100*p:.0f}% of similar cases.')
    return '\n'.join(lines)

## STEP 3 — Prompt conditioner functions

Builds the soft diagnostic prior from the rules table and injects it into the prompt.

In [ ]:
_INDICATION_KEYWORDS = {
    'Enlarged Cardiomediastinum': ['mediastinal widening', 'mediastinum', 'widened mediastinum'],
    'Cardiomegaly':     ['cardio', 'cardiac', 'heart', 'cardiomeg'],
    'Lung Opacity':     ['opacity', 'infiltrate', 'airspace disease', 'haziness'],
    'Lung Lesion':      ['nodule', 'mass', 'lesion', 'lung lesion', 'pulmonary nodule'],
    'Edema':            ['edema', 'pulmonary edema', 'fluid overload', 'chf'],
    'Consolidation':    ['consolidation', 'lobar consolidation', 'airspace consolidation'],
    'Pneumonia':        ['pneumonia', 'infection', 'fever', 'sepsis'],
    'Atelectasis':      ['atelectasis', 'collapse', 'subsegmental'],
    'Pneumothorax':     ['pneumothorax', 'ptx', 'chest pain', 'trauma'],
    'Pleural Effusion': ['effusion', 'pleural', 'pleural fluid'],
    'Pleural Other':    ['pleural thickening', 'pleural disease', 'empyema'],
    'Fracture':         ['fracture', 'rib fracture', 'trauma', 'fall'],
    'Support Devices':  ['intubated', 'icu', 'ventilated', 'post-op', 'line', 'tube'],
}

_NEGATION_PREFIXES = (
    'rule out', 'r/o', 'no ', 'without ', 'negative for', 'exclude ',
    'unlikely', 'no evidence', 'no signs', 'no sign of',
)

def infer_likely_labels(indication: str) -> list[str]:
    indication_lower = indication.lower()
    triggered = []
    for label, kws in _INDICATION_KEYWORDS.items():
        for kw in kws:
            idx = indication_lower.find(kw)
            if idx == -1:
                continue
            prefix = indication_lower[max(0, idx - 25):idx]
            if any(neg in prefix for neg in _NEGATION_PREFIXES):
                continue
            triggered.append(label)
            break
    return triggered

def build_association_hint(
    likely_labels: list[str],
    rules_df: pd.DataFrame,
    min_confidence: float = 0.25,
    min_lift: float = 1.2,
    max_rules: int = 4,
) -> str:
    """Return a formatted soft prior, or empty string if no strong rules apply."""
    if not likely_labels:
        return ''
    relevant = rules_df[
        rules_df['antecedent'].isin(likely_labels) &
        (rules_df['confidence'] >= min_confidence) &
        (rules_df['lift'] >= min_lift)
    ].sort_values('confidence', ascending=False).head(max_rules)

    if relevant.empty:
        return ''
    lines = ['Clinical association context (training set statistics):']
    for _, rule in relevant.iterrows():
        lines.append(
            f'  - {rule["antecedent"]} co-occurs with {rule["consequent"]} '
            f'in {100*rule["confidence"]:.0f}% of similar studies '
            f'(lift={rule["lift"]:.2f}).'
        )
    return '\n'.join(lines)

def build_conditioned_prompt(
    indication: str,
    rules_df: pd.DataFrame,
    min_confidence: float = 0.25,
    min_lift: float = 1.2,
    system_prompt: str = '',
) -> str:
    likely_labels = infer_likely_labels(indication)
    rule_hint     = build_association_hint(likely_labels, rules_df, min_confidence, min_lift)

    # TF-IDF label prior overrides rule hint when retrieval succeeds
    priors    = retrieve_label_prior(indication)
    prior_hint = build_label_prior_hint(priors) if priors is not None else ''
    hint = prior_hint if prior_hint else rule_hint

    parts = []
    if system_prompt:
        parts.append(system_prompt)
    if hint:
        parts.append(hint)
        parts.append('')
    if indication.strip():
        parts.append(f'Indication: {indication.strip()}')
    parts.append('Findings:')
    return '\n'.join(parts)

# ── Smoke test ────────────────────────────────────────────────────────────────
TEST_CASES = [
    'Shortness of breath, rule out pleural effusion',
    'Chest pain, history of CHF and pulmonary edema',
    'Post-op day 1, patient intubated in ICU',
    'Fever, cough, possible pneumonia',
    'Routine PA and lateral, no specific indication',
]
for ind in TEST_CASES:
    triggered = infer_likely_labels(ind)
    hint      = build_association_hint(triggered, rules_df)
    print(f'=== Indication: "{ind}"')
    print(f'    Triggered labels: {triggered}')
    if hint:
        print(hint)
    else:
        print('    (no strong rules — standard prompt)')
    print()

## STEP 4 — Conditioned inference on test set

Injects the association rule hint into every test prompt where the indication triggers a rule.
Cached to `reports/eval_hypotheses_assoc_rules_{VARIANT}.json`.

In [ ]:
from PIL import Image
from tqdm import tqdm
from peft import PeftModel
from src.training.model import load_model_and_processor

IMAGES_DIR = REPO_ROOT / params['data']['images_dir'] / 'images_normalized'
_BLANK = Image.new('RGB', (224, 224), color=(128, 128, 128))

SYSTEM_PROMPT = (
    'You are an expert radiologist. '
    'Write only the Findings section of a radiology report for the chest X-ray shown. '
    'Be concise and clinical. Do not include an Impression section.'
)

COND_VARIANT   = 'uniform_v3'
COND_MIN_CONF  = 0.25
COND_MIN_LIFT  = 1.2
variant_key    = f'assoc_rules_{COND_VARIANT}'
cache_hyps     = REPO_ROOT / 'reports' / f'eval_hypotheses_{variant_key}.json'

def _load_image(row):
    frontal = row.get('frontal', [])
    if hasattr(frontal, '__len__') and len(frontal) > 0:
        p = IMAGES_DIR / list(frontal)[0]
        if p.exists():
            try: return Image.open(p).convert('RGB')
            except Exception: pass
    return _BLANK

if cache_hyps.exists():
    cond_hyps = json.loads(cache_hyps.read_text())
    print(f'Loaded {len(cond_hyps)} cached hypotheses from {cache_hyps.name}')
else:
    print(f'Loading {COND_VARIANT} for conditioned inference…')
    model, processor = load_model_and_processor(
        model_id=params['model']['base_model_id'],
        quantization=params['model']['quantization'],
    )
    adapter_path = CHECKPOINT_DIR / COND_VARIANT / 'best_model'
    if not adapter_path.exists():
        adapter_path = CHECKPOINT_DIR / f'qlora_{COND_VARIANT}' / 'best_model'
    model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()

    cond_hyps = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=variant_key):
        indication = str(row.get('indication', '') or '').strip()
        if indication.lower() in {'nan', 'none', ''}: indication = ''

        user_text = build_conditioned_prompt(
            indication, rules_df,
            min_confidence=COND_MIN_CONF,
            min_lift=COND_MIN_LIFT,
            system_prompt=SYSTEM_PROMPT,
        )
        img = _load_image(row)
        content = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
        prompt = processor.apply_chat_template(
            [{'role': 'user', 'content': content}],
            add_generation_prompt=True, tokenize=False,
        )
        inputs = processor(text=prompt, images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model.generate(**inputs,
                max_new_tokens=params['model']['max_new_tokens'],
                do_sample=False, pad_token_id=processor.tokenizer.eos_token_id)
        hyp = processor.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        cond_hyps.append(hyp)

    cache_hyps.write_text(json.dumps(cond_hyps, ensure_ascii=False, indent=2))
    del model, processor; torch.cuda.empty_cache()
    print(f'Done. {len(cond_hyps)} hypotheses saved.')

n_triggered = sum(
    1 for _, row in test_df.iterrows()
    if infer_likely_labels(str(row.get('indication', '') or '').strip())
)
print(f'Rules triggered in {n_triggered}/{len(test_df)} studies ({100*n_triggered/len(test_df):.1f}%).')

## STEP 5 — Compute and save metrics

In [ ]:
import bert_score.utils as _bsu
from bert_score import score as _bert_score
from src.data.labels import run_chexbert
from sklearn.metrics import f1_score
import evaluate as hf_evaluate

_orig_sent_encode = _bsu.sent_encode
def _safe_sent_encode(tokenizer, sent):
    if getattr(tokenizer, 'model_max_length', 0) > 10_000:
        tokenizer.model_max_length = 512
    return _orig_sent_encode(tokenizer, sent)
_bsu.sent_encode = _safe_sent_encode

_bleu  = hf_evaluate.load('bleu')
_rouge = hf_evaluate.load('rouge')

cache_m = REPO_ROOT / 'reports' / f'eval_metrics_{variant_key}.json'
if cache_m.exists():
    cond_metrics = json.loads(cache_m.read_text())
    print(f'Loaded metrics from {cache_m.name}')
else:
    refs = references
    print('BERTScore…')
    _, _, F = _bert_score(cond_hyps, refs,
        model_type=params['eval']['bertscore_model'],
        lang='en', device=DEVICE, verbose=False, batch_size=16)
    print('CheXbert…')
    hyp_labels = run_chexbert(cond_hyps, uncertain_policy='present', device=DEVICE)
    ref_labels = run_chexbert(refs,      uncertain_policy='present', device=DEVICE)
    print('BLEU/ROUGE…')
    cond_metrics = {
        'variant': variant_key,
        'bertscore_f1':      float(F.mean()),
        'chexbert_micro_f1': float(f1_score(ref_labels, hyp_labels, average='micro', zero_division=0)),
        'chexbert_macro_f1': float(f1_score(ref_labels, hyp_labels, average='macro', zero_division=0)),
        'bleu4':             _bleu.compute(predictions=cond_hyps, references=[[r] for r in refs], max_order=4)['bleu'],
        'rouge_l':           _rouge.compute(predictions=cond_hyps, references=refs)['rougeL'],
        'per_label_f1':      {label: float(f1_score(ref_labels[:, i], hyp_labels[:, i], zero_division=0))
                              for i, label in enumerate(CHEXBERT_LABELS)},
        'bertscore_per_study': F.tolist(),
    }
    cache_m.write_text(json.dumps(cond_metrics, indent=2))
    print(f'Saved → {cache_m.name}')

_bsu.sent_encode = _orig_sent_encode

# Load baseline for comparison
baseline_v3_path = REPO_ROOT / 'reports' / 'eval_metrics_uniform_v3.json'
comparison = {}
if baseline_v3_path.exists():
    comparison['uniform_v3'] = json.loads(baseline_v3_path.read_text())
comparison[variant_key] = cond_metrics

DISPLAY_NAMES = {
    'uniform_v3':              'QLoRA uniform (v3) — no conditioner',
    'assoc_rules_uniform_v3':  'Assoc. rules conditioner (v3)',
}

rows = []
for v, m in comparison.items():
    rows.append({'Model': DISPLAY_NAMES.get(v, v),
                 'BERTScore-F1':      round(m['bertscore_f1'],      4),
                 'CheXbert micro-F1': round(m['chexbert_micro_f1'], 4),
                 'CheXbert macro-F1': round(m['chexbert_macro_f1'], 4),
                 'BLEU-4':            round(m['bleu4'],             4),
                 'ROUGE-L':           round(m['rouge_l'],           4)})
display(pd.DataFrame(rows).set_index('Model'))

if 'uniform_v3' in comparison:
    delta = cond_metrics['bertscore_f1'] - comparison['uniform_v3']['bertscore_f1']
    print(f'\nΔ BERTScore vs no-conditioner baseline: {delta:+.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
names  = [DISPLAY_NAMES.get(v, v) for v in comparison]
bscore = [comparison[v]['bertscore_f1'] for v in comparison]
colors = ['#1f77b4', '#ff7f0e']
bars   = ax.bar(names, bscore, color=colors[:len(comparison)], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, bscore):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylim(min(bscore) * 0.98, max(bscore) * 1.04)
ax.set_ylabel('BERTScore-F1')
ax.set_title('Association Rules Conditioner — impact on BERTScore-F1')
ax.tick_params(axis='x', labelrotation=15)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'eval_assoc_rules_comparison.png', dpi=150, bbox_inches='tight')
print('Saved eval_assoc_rules_comparison.png'); plt.show()

## STEP 6 — Per-indication analysis

Checks whether the conditioner helps specifically on the studies where a rule was triggered.

In [ ]:
baseline_v3_path = REPO_ROOT / 'reports' / 'eval_metrics_uniform_v3.json'
if not baseline_v3_path.exists():
    print('Run notebook 04 first to generate eval_metrics_uniform_v3.json')
else:
    base_scores  = np.array(json.loads(baseline_v3_path.read_text())['bertscore_per_study'])
    cond_scores  = np.array(cond_metrics['bertscore_per_study'])

    triggered_mask = np.array([
        bool(infer_likely_labels(str(row.get('indication', '') or '').strip()))
        for _, row in test_df.iterrows()
    ])

    n_trig = triggered_mask.sum()
    n_notrig = (~triggered_mask).sum()

    print(f'Studies with rule triggered : {n_trig} ({100*n_trig/len(test_df):.1f}%)')
    print(f'Studies without rule         : {n_notrig} ({100*n_notrig/len(test_df):.1f}%)')
    print()
    print('BERTScore — triggered subset:')
    print(f'  Baseline (v3)  : {base_scores[triggered_mask].mean():.4f}')
    print(f'  With rules     : {cond_scores[triggered_mask].mean():.4f}')
    print(f'  Δ              : {(cond_scores - base_scores)[triggered_mask].mean():+.4f}')
    print()
    print('BERTScore — non-triggered subset:')
    print(f'  Baseline (v3)  : {base_scores[~triggered_mask].mean():.4f}')
    print(f'  With rules     : {cond_scores[~triggered_mask].mean():.4f}')
    print(f'  Δ              : {(cond_scores - base_scores)[~triggered_mask].mean():+.4f}')

    # Delta distribution
    delta_scores = cond_scores - base_scores
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(delta_scores[triggered_mask],  bins=40, alpha=0.7, label=f'Triggered (n={n_trig})',  color='#ff7f0e')
    ax.hist(delta_scores[~triggered_mask], bins=40, alpha=0.7, label=f'Not triggered (n={n_notrig})', color='#1f77b4')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('ΔBERT Score (conditioned − baseline)')
    ax.set_ylabel('Count')
    ax.set_title('Per-study BERTScore delta — conditioner vs baseline')
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'eval_assoc_rules_delta.png', dpi=150, bbox_inches='tight')
    print('Saved eval_assoc_rules_delta.png'); plt.show()

## Done

Metrics written to `reports/eval_metrics_assoc_rules_uniform_v3.json`.  
**Next:** return to notebook 04 STEP 7 and run the grand comparison cell — it will auto-discover
all `eval_metrics_*.json` files including RAG (05) and association rules (06).